In [ ]:
USE DATABASE database_name;
use schema schema_name;

In [ ]:
CREATE OR REPLACE Table schema_name.silver_customer AS
WITH first_orders AS (
    SELECT
        customer_id,
        CAST(MIN(order_time) AS DATE) AS first_order_date
    FROM schema_name.bronze_orders
    WHERE order_time IS NOT NULL
    GROUP BY customer_id
)
SELECT
    c.customer_id      AS customer_id,
    c.first_name       AS first_name,
    c.last_name        AS last_name,
    c.email_address As email_address,
    c.date_of_birth    AS date_of_birth,
    fo.first_order_date AS first_order_date,
    c.membership_type  AS membership_type
FROM schema_name.bronze_customer AS c
LEFT JOIN first_orders AS fo
  ON fo.customer_id = c.customer_id;

In [ ]:
CREATE OR REPLACE TABLE schema_name.SILVER_ORDERS AS
WITH riders AS (
  SELECT
    r.RIDER_ID AS RIDER_ID_NUM,
    r.PRIMARY_DELIVERY_METHOD
  FROM schema_name.BRONZE_RIDER AS r
),
orders AS (
  SELECT
    CAST(o.ORDER_ID      AS NUMBER(19,0)) AS ORDER_ID,
    CAST(o.CUSTOMER_ID   AS NUMBER(19,0)) AS CUSTOMER_ID,
    CAST(o.RESTAURANT_ID AS NUMBER(19,0)) AS RESTAURANT_ID,
    CAST(o.RIDER_ID      AS NUMBER(19,0)) AS RIDER_ID,
    o.IDX,
    o.STATE,
    o.ORDER_TOTAL,
    o.DISCOUNT,
    o.ORDER_TIME,
    o.PICKUP_TIME,
    o.DELIVERY_TIME,
    o.DEVICE_TYPE,
    o.RIDER_PAYMENT,
    o.RIDER_TIP,
    DATEDIFF('minute', o.ORDER_TIME, o.DELIVERY_TIME) AS ORDER_TOTAL_MINUTES,
    CONCAT(TO_VARCHAR(o.DELIVERY_LATITUDE), ',', TO_VARCHAR(o.DELIVERY_LONGITUDE)) AS DELIVERY_COORDS
  FROM schema_name.BRONZE_ORDERS AS o
)
SELECT
  o.ORDER_ID,
  o.CUSTOMER_ID,
  o.RESTAURANT_ID,
  o.RIDER_ID,
  o.IDX,
  o.STATE,
  o.ORDER_TOTAL,
  o.DISCOUNT,
  o.ORDER_TIME,
  o.PICKUP_TIME,
  o.DELIVERY_TIME,
  o.DEVICE_TYPE,
  COALESCE(r.PRIMARY_DELIVERY_METHOD, 'bicycle') AS RIDER_VEHICLE,
  o.RIDER_PAYMENT,
  o.RIDER_TIP,
  o.ORDER_TOTAL_MINUTES,
  o.DELIVERY_COORDS
FROM orders AS o
LEFT JOIN riders AS r
  ON o.RIDER_ID = r.RIDER_ID_NUM;

In [ ]:
# Creating silver rider dataframe

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, call_builtin, lit

session = get_active_session()

query = "SELECT * FROM database_name.schema_name.bronze_rider"

silverriderdf = session.sql(query)

# Setting join_date column to correct date format

dateformat = 'MON DD YYYY HH12:MIAM'

silverriderdf = silverriderdf.with_column(
    "join_date",
    call_builtin("COALESCE",
                 call_builtin("TRY_TO_DATE", col("join_date"), dateformat),
                 call_builtin("TRY_TO_DATE", col("join_date"))))

# Default NULLs in primary_delivery_method to 'bicycle'

silverriderdf = silverriderdf.na.fill({"primary_delivery_method":'bicycle'})

# Remove leading zeros in addresses

silverriderdf = silverriderdf.with_column(
    "address",
    call_builtin("LTRIM", col("address"), lit("0")))

# Create table

silverriderdf.write.mode("overwrite").save_as_table("silver_rider")

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, when, lit, concat, to_varchar
)

session = get_active_session()

# Load BRONZE table
restaurant_silver = session.table("database_name.schema_name.bronze_restaurant")

# Add concatenated coordinate column
restaurant_silver = restaurant_silver.with_column(
    "COMPLETE_COORDS",
    concat(
        to_varchar(col("LATITUDE")),
        lit(","),
        to_varchar(col("LONGITUDE"))
    )
)


restaurant_silver = restaurant_silver.with_column(
    "ZIP_CODE",
    col("ZIP_CODE").cast("INT")
)

# Clean 'city' field
restaurant_silver = restaurant_silver.with_column(
    "city",
    when(
        (col("city") == lit("New York")) | (col("city") == lit("New York City")),
        lit("Manhattan")
    )
    .when(
        ~col("city").isin(["Manhattan", "Queens", "Staten Island", "Bronx", "Brooklyn"]),
        lit("Queens")
    )
    .otherwise(col("city"))
)

# Rename city → borough AFTER cleaning
restaurant_silver = restaurant_silver.with_column_renamed("city", "borough")
restaurant_silver = restaurant_silver.with_column_renamed("brand_name", "restaurant_name")


# Drop unwanted columns
restaurant_silver = restaurant_silver.drop(
    "serves_halal",
    "kid_friendly",
    "has_tv",
    "wheelchair_accessible",
    "has_takeout",
    "has_smoking_area",
    "accepts_credit_cards",
    "offers_pickup",
    "serves_gluten_free",
    "serves_vegan",
    "has_outdoor_seating",
    "serves_kosher",
    "observed_holidays",
    "has_bar",
    "serves_keto",
    "serves_vegetarian",
    "LATITUDE",
    "LONGITUDE",
    "SERVES_DINNER",
    "SERVES_LUNCH",
    "HAS_DELIVERY",
    "OFFERS_DELIVERY",
    "SERVES_HALA",
    "INSTAGRAM",
    "TWITTER",
    "FACEBOOK",
    "RESTAURANT_WEBSITE",
    "EMAIL_ADDRESS",
    "DESCRIPTION",
    "TELEPHONE_NUMBER",
    "COUNTRY",
    "SUBREGION",
    "ADDRESS"
)

In [ ]:
pdf = restaurant_silver.to_pandas()

import re
import math

def normalize_hours(s: str) -> str | None:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return None

    original_txt = str(s).strip()
    if original_txt == "":
        return None

    txt = original_txt
    lower = txt.lower().strip()

    # Canonical quick cases
    if "open 24" in lower:
        return "00:00-24:00"
    if "closed" in lower:
        return "CLOSED"

    # Normalize separators and spacing
    lower = re.sub(r'[–—]', '-', lower)
    lower = re.sub(r'\s+', ' ', lower).strip()

    def parse_time_token(token: str):
        """Return (hh:mm, total_minutes) for comparison."""
        t = token.strip().lower()
        t = t.replace('.', '')
        t = t.replace('noon', '12:00 pm').replace('midnight', '12:00 am')

        # Add :00 if only hour + am/pm
        t = re.sub(r"^(\d{1,2})\s*(am|pm)$", r"\1:00 \2", t)

        # AM/PM
        m = re.match(r"^(\d{1,2})(?::(\d{2}))?\s*(am|pm)$", t)
        if m:
            h = int(m.group(1))
            mi = int(m.group(2) or 0)
            ap = m.group(3).lower()
            if ap == "am":
                if h == 12:  # midnight
                    h = 0
            else:
                if h != 12:
                    h += 12
            return f"{h:02d}:{mi:02d}", h * 60 + mi

        # 24h HH:MM
        m2 = re.match(r"^(\d{1,2}):(\d{2})$", t)
        if m2:
            h = int(m2.group(1))
            mi = int(m2.group(2))
            if 0 <= h <= 24 and 0 <= mi <= 59:
                return f"{h:02d}:{mi:02d}", (0 if h == 24 else h) * 60 + mi

        # Not parseable
        return None, None

    parts = [p.strip() for p in re.split(r"\s*,\s*", lower) if p.strip()]
    normalized_parts = []

    for part in parts:
        explicit_next_day = 'next day' in part
        part = part.replace("(next day)", "").strip()

        # Split time ranges
        rng = re.split(r"\s*-\s*", part)
        if len(rng) != 2:
            return original_txt

        start_raw, end_raw = rng
        s24, smin = parse_time_token(start_raw)
        e24, emin = parse_time_token(end_raw)

        if not s24 or not e24:
            return original_txt

        # Determine if overnight
        overnight = explicit_next_day or (emin < smin)

        # Format end time
        if overnight:
            if e24 == "00:00":
                end_fmt = "24:00"  # midnight end-of-day
            else:
                end_fmt = f"{e24}"  # next day - add plus one to here if want to show its next day
        else:
            end_fmt = e24

        normalized_parts.append(f"{s24}-{end_fmt}")

    return "; ".join(normalized_parts)


hours_cols = [
    "OPERATING_HOURS_MON",
    "OPERATING_HOURS_TUE",
    "OPERATING_HOURS_WED",
    "OPERATING_HOURS_THU",
    "OPERATING_HOURS_FRI",
    "OPERATING_HOURS_SAT",
    "OPERATING_HOURS_SUN",
]

for c in hours_cols:
    pdf[c] = pdf[c].apply(normalize_hours)

    session.write_pandas(pdf, "SILVER_RESTAURANT", overwrite=True)
    

In [ ]:
CREATE OR REPLACE TABLE database_name.schema_name.SILVER_ORDERS AS
SELECT
    o.*,
    HAVERSINE(
        STRTOK(o.delivery_coords, ',', 1)::FLOAT,
        STRTOK(o.delivery_coords, ',', 2)::FLOAT,
        STRTOK(r.complete_coords, ',', 1)::FLOAT,
        STRTOK(r.complete_coords, ',', 2)::FLOAT
    ) * 0.621371 AS total_miles_travelled
FROM database_name.schema_name.silver_orders AS o
LEFT JOIN database_name.schema_name.silver_restaurant AS r
    ON o.restaurant_id = r.restaurant_id;